[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/choROPeNt/FFTjax/blob/main/notebooks/in-elastic_J2.ipynb)

# J2 (von Mises) Elastoplasticity — Quad RVE with a Plastic Matrix

FFTjax's rate-independent J2 plasticity material model
(`materialmodels.inelastic.plasticity_j2.J2Plasticity`), linear isotropic
hardening, demonstrated on a real two-phase composite RVE. The tangent
stiffness is obtained via automatic differentiation rather than a
hand-derived closed-form expression.

## Theoretical background

Yield function (von Mises, isotropic hardening):

$$
f(\boldsymbol{\sigma}, \alpha) = q(\boldsymbol{\sigma}) - \big(\sigma_{y0} + H\alpha\big),
\qquad
q(\boldsymbol{\sigma}) = \sqrt{\tfrac{3}{2}\, \mathbf{s}:\mathbf{s}}, \quad
\mathbf{s} = \boldsymbol{\sigma} - \tfrac{1}{3}\mathrm{tr}(\boldsymbol{\sigma})\,\mathbf{I}
$$

$\sigma_{y0}$ is the initial (virgin) yield stress and $H$ the constant
isotropic hardening modulus -- yielding begins once the von Mises
equivalent stress $q$ reaches the current yield surface
$\sigma_{y0} + H\alpha$, where $\alpha$ is the accumulated equivalent
plastic strain.

Associative flow, closed-form radial return (linear hardening -- a
nonlinear hardening law, e.g. Voce, would need a local scalar Newton solve
for the plastic multiplier $\Delta\gamma$ instead of the closed form below):

$$
\Delta\gamma = \frac{\langle f_\text{trial}\rangle}{3\mu + H}, \qquad
\mathbf{s} = \Big(1 - \frac{3\mu\,\Delta\gamma}{q_\text{trial}}\Big)\mathbf{s}_\text{trial}, \qquad
\boldsymbol{\varepsilon}_p \mathrel{+}= \tfrac{3}{2}\Delta\gamma\,\frac{\mathbf{s}_\text{trial}}{q_\text{trial}}
$$

where the trial state $(\mathbf{s}_\text{trial}, q_\text{trial})$ is the
elastic predictor evaluated at the previous plastic strain
$\boldsymbol{\varepsilon}_p$; $\langle x \rangle = \max(x, 0)$ makes the
plastic multiplier -- and with it the whole update -- collapse to the
elastic identity whenever the trial state is still inside the yield
surface, with no separate branch needed.

Unlike the linear elastic models, this material is **stateful**: stress and
tangent depend on $(\boldsymbol{\varepsilon}, \boldsymbol{\varepsilon}_p, \alpha)$, not just
$\boldsymbol{\varepsilon}$ -- the plastic strain $\boldsymbol{\varepsilon}_p$ and accumulated
equivalent plastic strain $\alpha$ are carried forward by the caller across
load steps, never stored on the material instance. The tangent itself is
derived by automatic differentiation of the stress update above, rather
than a hand-derived closed-form expression:

$$
\mathbb{C}^\text{tan}_{ijkl} = \frac{\partial \sigma_{ij}}{\partial \varepsilon_{kl}}
\Big(\boldsymbol{\varepsilon}, \boldsymbol{\varepsilon}_p, \alpha\Big)
$$

computed with `jax.jacfwd` on the stress update -- one autodiff pass
returns the tangent, the stress, and the updated state together:

```python
def stress_and_tangent(self, eps, eps_p_prev, alpha_prev):
    def _fn(e):
        sigma, new_state = self.stress(e, eps_p_prev, alpha_prev)
        return sigma, (sigma, new_state)          # stress rides along as aux
    C_tan, (sigma, new_state) = jax.jacfwd(_fn, has_aux=True)(eps)
    return sigma, C_tan, new_state
```

This notebook demonstrates the model on a square-packed glass-fiber/epoxy
RVE -- the same geometry as `notebooks/lin-elastic_strain.ipynb` -- with a
plastic epoxy matrix, showing the macroscopic (homogenized) nonlinearity
that emerges purely from the local constitutive law once enough of the
matrix has yielded.

## Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running on Colab — installing FFTjax...")
    %pip install -q git+https://github.com/choROPeNt/FFTjax.git
else:
    import sys
    sys.path.insert(0, "../src")
    print("Running locally — using the local src/ checkout.")

import utils.precision  # side effect: configures JAX (X64 off on TPU, no GPU prealloc)
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

print("JAX backend:", jax.default_backend())
print("float64 enabled:", jnp.zeros(1).dtype == jnp.float64)


## RVE demonstration

Because the matrix's stiffness now depends on the (unknown) strain and its
own history, this can't go through the plain linear solvers
(`solve_lippmann_schwinger`/`solve_displacement_based`) the way the elastic
notebook does -- those assume a fixed per-voxel `C_field` for the whole CG
solve. Instead this uses a Newton-outer/CG-inner driver,
`problems.mechanics.solve_displacement_based_nonlinear`: at each Newton
iteration it evaluates the actual (nonlinear) stress and tangent at the
current strain guess via a per-voxel `local_update` callable, and uses CG
only for the linearized correction -- true Newton, not a fixed-point
scheme. It lives in `problems/mechanics.py` rather than `solvers/` for the
same reason `problems.fracture`'s staggered loop does (see that module's
docstring): it's an iterative scheme built from repeated solver calls, not
itself a reusable numerical primitive. See
`test/test_problems_mechanics_nonlinear.py` for its validation -- including
reproducing `solve_displacement_based`'s own answer to ~1e-13 when given a
plain linear `local_update`, the strongest available check that the new
driver itself is correct before trusting it on a genuinely nonlinear
problem.

### Generate the composite RVE

Same generator/parameters as the elastic notebook, with a slightly coarser
grid (`N_min=24` vs. 32) so the load-stepped Newton-CG solve below stays
fast for a demo.

In [ ]:
from generation.rve import make_square_composite_rve

phi     = 0.5      # target fiber volume fraction
r_fiber = 0.005     # fiber radius [mm]
dx      = 0.0002    # target voxel size [mm]

phase_np, n, L, phi_act = make_square_composite_rve(
    phi=phi, r_fiber=r_fiber, dx=dx, N_min=24, nz=1,
)
Nv = int(np.prod(n))
phase = jnp.array(phase_np.reshape(-1))

print("grid n :", n)
print("total voxels Nv:", Nv)
print("fiber volume fraction (actual):", f"{phi_act:.4f}")


In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(phase_np[:, :, 0].T, origin="lower", cmap="gray_r",
          extent=(0, n[0] * dx * 1000, 0, n[1] * dx * 1000))
ax.set_title(f"Fiber cross-section (Vf={phi_act:.3f})")
ax.set_xlabel("x [µm]")
ax.set_ylabel("y [µm]")
ax.set_aspect("equal")
plt.show()


### Materials

Fiber: same glass fiber as the elastic notebook. Matrix: `J2Plasticity` with
a modest yield stress, chosen so the demo clearly engages the plastic
branch at an easily-reached strain level -- not a literature epoxy yield
stress.

`local_update` is the per-voxel combinator this new driver needs: for fiber
voxels it's the plain linear `sigma = C:eps` (constant tangent, no state);
for matrix voxels it's `J2Plasticity.stress_and_tangent_field`. Combined by
phase via `jnp.where`, the same mixed constant/per-voxel-field pattern
`materialmodels.assembly.assemble_C_field` already uses for a per-voxel
`fiber_dir`.

In [ ]:
from materialmodels.elastic.isotropic import LinearElasticIsotropic
from materialmodels.inelastic.plasticity_j2 import J2Plasticity

fiber  = LinearElasticIsotropic(E=70.0e3, nu=0.20, name="glass fiber")
matrix = J2Plasticity(E=3.76e3, nu=0.39, sigma_y0=50.0, H=1.0e3, name="epoxy matrix (plastic)")
C_fiber = fiber.stiffness_tensor()

print(fiber)
print(matrix)


def local_update(eps_field, state):
    # eps_field (3,3,Nv), state=(eps_p, alpha) -> (sigma, C_tan, new_state)
    eps_p_field, alpha_field = state
    sigma_pl, C_pl, (eps_p_new, alpha_new) = matrix.stress_and_tangent_field(
        eps_field, eps_p_field, alpha_field
    )
    sigma_el = jnp.einsum("ijkl,klm->ijm", C_fiber, eps_field)
    C_el     = jnp.broadcast_to(C_fiber[..., None], C_pl.shape)

    is_matrix = (phase == 0)
    sigma     = jnp.where(is_matrix, sigma_pl, sigma_el)
    C_tan     = jnp.where(is_matrix, C_pl, C_el)
    eps_p_out = jnp.where(is_matrix, eps_p_new, eps_p_field)
    alpha_out = jnp.where(is_matrix, alpha_new, alpha_field)
    return sigma, C_tan, (eps_p_out, alpha_out)


### Solve: load / unload / reload cycle, homogenized shear response

Ramp the macroscopic shear strain $\bar\varepsilon_{12}$ from 0 up past the
matrix's yield strain, unload all the way through zero into reversed
(compressive) shear, then reload back up -- carrying `(eps_p, alpha)`
forward from one step to the next throughout, the same "state carried by
the caller across increments" convention `problems.fracture` already uses
for its `d`/`H` damage state. Plotting the homogenized (volume-averaged)
shear stress against the applied strain traces out a **hysteresis loop** at
the RVE (composite) scale -- elastic unloading (unloading slope = the
elastic modulus, not the plastic tangent) and a permanent offset between
the loading and reloading branches -- even though only the matrix phase
itself is plastic; the fiber stays purely elastic throughout.

Each converged load step is also written as one increment to
`output/notebooks/rve_plastic_matrix.xdmf`/`.h5` via `IncrementalWriter`
(the project-wide standard for field-data output) -- open the `.xdmf` in
ParaView (`Xdmf3ReaderT`) to scrub through the whole load/unload/reload
history and watch the plastic zone grow, then stay locked in, around the
fiber.

In [ ]:
import os

from operators.green import build_freq_grid
from post.fields import compute_displacement, field_to_grid, to_voigt, von_mises
from problems.mechanics import solve_displacement_based_nonlinear
from utils.io.xdmf_writer import IncrementalWriter

xi_flat = build_freq_grid(n, L)

gamma_max = 0.03
n_load, n_unload, n_reload = 10, 15, 15
gammas_load   = np.linspace(0.0, gamma_max, n_load + 1)[1:]
gammas_unload = np.linspace(gamma_max, -gamma_max, n_unload + 1)[1:]
gammas_reload = np.linspace(-gamma_max, gamma_max, n_reload + 1)[1:]
# prepend the virgin gamma=0 state -- trivially eps=sigma=0, no Newton solve
# needed, but without it explicitly here both the plot and the XDMF export
# jump straight to the first small increment, skipping the true (0, 0)
# starting point of the hysteresis loop.
gammas_applied = np.concatenate([[0.0], gammas_load, gammas_unload, gammas_reload])
# segment boundaries for the loading/unloading/reloading plot below (+1 for
# the prepended gamma=0 state, grouped into the "load" segment)
i_load_end, i_unload_end = 1 + len(gammas_load), 1 + len(gammas_load) + len(gammas_unload)

output_dir = "output" if IN_COLAB else "../output/notebooks"
os.makedirs(output_dir, exist_ok=True)

state = (jnp.zeros((3, 3, Nv)), jnp.zeros(Nv))
eps_prev_full = jnp.zeros((3, 3, Nv))  # warm-start seed: zero strain at gamma=0
tau_avg_path, n_iters_path = [], []
with IncrementalWriter(f"{output_dir}/rve_plastic_matrix", grid_shape=n, grid_length=L) as writer:
    for step, gamma in enumerate(gammas_applied):
        eps_bar_step = jnp.array([[0., gamma / 2, 0.], [gamma / 2, 0., 0.], [0., 0., 0.]])
        if step == 0:
            # virgin state: eps_bar_step is already zero here, so this is just
            # the known trivial solution -- skip the Newton solve entirely.
            eps_step, sigma_step = jnp.zeros((3, 3, Nv)), jnp.zeros((3, 3, Nv))
        else:
            eps0_step = jnp.ones((3, 3, Nv)) * eps_bar_step[:, :, None]
            # warm start: previous step converged eps, minus this step's new baseline --
            # without this, Newton starts every step from a zero-fluctuation guess,
            # which gets a progressively worse starting point past yield (verified:
            # without warm starting, this exact sweep fails to converge partway through).
            # Works the same way whether gamma is increasing (loading/reloading) or
            # decreasing (unloading) -- only the step-to-step change matters, not
            # the overall direction of the path.
            delta_init = eps_prev_full - eps0_step

            eps_step, sigma_step, state, converged, n_iter = solve_displacement_based_nonlinear(
                n, xi_flat, eps_bar_step, local_update, state,
                toler_lin=1e-7, maxiter_lin=2000, toler_nr=1e-7, maxiter_nr=50,
                delta_init=delta_init,
            )
            assert converged, f"Newton did not converge at gamma={gamma:.4f}"
            n_iters_path.append(n_iter)
        eps_prev_full = eps_step
        tau_avg_path.append(float(jnp.mean(sigma_step[0, 1])))

        _, alpha_step = state
        eps_grid   = field_to_grid(eps_step, n)
        sigma_grid = field_to_grid(sigma_step, n)
        u_grid     = compute_displacement(eps_step, eps_bar_step, n, L)
        writer.write_increment(step, {
            "phase":        phase_np.astype(np.float64),
            "displacement": u_grid.astype(np.float64),   # cell array, matching lin-elastic_strain.ipynb
            "strain":       to_voigt(eps_grid).astype(np.float64),
            "stress":       to_voigt(sigma_grid).astype(np.float64),
            "von_mises":    von_mises(sigma_grid).astype(np.float64),
            "strain_p":     np.array(alpha_step).reshape(n).astype(np.float64),  # accumulated equivalent plastic strain (alpha)
        }, time=float(step))  # step index, not gamma -- gamma reverses direction during
        # unload/reload, so it isn't a valid (monotonic) XDMF time axis; duplicate/
        # non-monotonic time values are silently mishandled by XDMF readers such as
        # ParaView (confirmed root cause of "missing field arrays" at specific steps).

print(f"Newton iterations per solved step: min={min(n_iters_path)}, max={max(n_iters_path)}")
print(f"Wrote {output_dir}/rve_plastic_matrix.h5")
print(f"      {output_dir}/rve_plastic_matrix.xdmf")

tau_avg_path = np.array(tau_avg_path)
plt.figure(figsize=(5.5, 4.5))
plt.plot(gammas_applied[:i_load_end], tau_avg_path[:i_load_end],
         'o-', ms=4, color="gray", label="load")
plt.plot(gammas_applied[i_load_end - 1:i_unload_end], tau_avg_path[i_load_end - 1:i_unload_end],
         's--', ms=4, color="gray", label="unload")
plt.plot(gammas_applied[i_unload_end - 1:], tau_avg_path[i_unload_end - 1:],
         '^:', ms=4, color="gray", label="reload")
plt.axhline(0, color='gray', lw=0.5)
plt.axvline(0, color='gray', lw=0.5)
plt.xlabel(r"applied shear strain $\bar\gamma_{12} = 2\bar\varepsilon_{12}$")
plt.ylabel(r"homogenized shear stress $\langle\sigma_{12}\rangle$ [MPa]")
plt.title("RVE hysteresis (elastic fiber + plastic matrix)")
plt.legend()
plt.tight_layout()
plt.savefig(f"{output_dir}/rve_plastic_matrix_hysteresis.pdf", dpi=300)
plt.show()


### Where does the matrix yield first?

The accumulated equivalent plastic strain (`strain_p` in the exported
fields, `alpha` internally) at the end of the full load/unload/reload
cycle, mapped back onto the voxel grid, shows where the matrix has actually
yielded -- physically expected to concentrate near the fiber/matrix
interface, where the stiff fiber locally raises the matrix's stress above
its far-field average. `strain_p` only ever grows (it's monotone by
construction, same as any accumulated equivalent plastic strain), so this
reflects yielding from *either* loading direction, not just the final
(reloaded) state.

In [ ]:
eps_p_final, alpha_final = state
strain_p_grid = np.array(alpha_final).reshape(n)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(phase_np[:, :, 0].T, origin="lower", cmap="gray_r")
axes[0].set_title("Fiber phase")
im = axes[1].imshow(strain_p_grid[:, :, 0].T, origin="lower", cmap="plasma")
axes[1].set_title("Accumulated plastic strain (strain_p)")
plt.colorbar(im, ax=axes[1], fraction=0.046)
for ax in axes:
    ax.set_xlabel("voxel x")
    ax.set_ylabel("voxel y")
plt.tight_layout()
plt.show()

print(f"voxels with strain_p > 0: {int(np.sum(strain_p_grid > 1e-12))} / {Nv}")
print(f"max strain_p: {float(np.max(strain_p_grid)):.4e}")


## Summary

- `J2Plasticity` (closed-form radial return, linear isotropic hardening, an
  autodiff-derived tangent rather than a hand-derived formula) plugs into an
  actual FFT solve via a new Newton-outer/CG-inner driver,
  `problems.mechanics.solve_displacement_based_nonlinear` -- needed because,
  unlike the linear elastic models, this material's tangent depends on the
  unknown strain and can't be assembled once up front.
- That driver was validated first by reproducing the known-good linear
  solver to ~1e-13 on a linear problem, then shown converging on the real
  two-phase RVE with a plastic matrix above, including through a full
  load/unload/reload cycle (warm-started from the previous step, robust
  whether the applied strain is increasing or decreasing).
- The RVE's homogenized response traces out a **hysteresis loop** at the
  composite scale -- elastic unloading and a permanent offset between the
  loading and reloading branches -- even though only the matrix phase is
  plastic, with a physically sensible spatial pattern of yielded matrix
  voxels concentrated near the fiber interface.